# VisualVocab YOLO Training
This notebook downloads the merged YOLO dataset from Render, trains YOLO11 Nano, exports TFLite, and downloads the model package.

In [ ]:
!pip install -q --upgrade ultralytics
from ultralytics import YOLO
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
DATASET_URL = 'https://visualvocab-backend.onrender.com/visualvocab/training/combined.zip'
print(DATASET_URL)

In [ ]:
from pathlib import Path
import requests, zipfile, shutil

zip_path = Path('/content/visualvocab_combined_yolo.zip')
response = requests.get(DATASET_URL, timeout=180)
response.raise_for_status()
zip_path.write_bytes(response.content)

dataset_root = Path('/content/visualvocab_dataset')
shutil.rmtree(dataset_root, ignore_errors=True)
dataset_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(dataset_root)

print('Dataset:', dataset_root)
print((dataset_root / 'dataset.yaml').read_text())

In [ ]:
model = YOLO('yolo11n.pt')

model.train(
    data=str(dataset_root / 'dataset.yaml'),
    epochs=50,
    imgsz=640,
    batch=8,
    project='/content/runs',
    name='visualvocab',
    pretrained=True,
)


In [ ]:
best_pt = Path('/content/runs/visualvocab/weights/best.pt')
trained = YOLO(str(best_pt))
export_result = trained.export(
    format='tflite',
    imgsz=640,
    int8=False,
    nms=False,
)
print('Export result:', export_result)

In [ ]:
from google.colab import files
import json

tflite_files = sorted(Path('/content').rglob('*.tflite'))
print('\n'.join(str(path) for path in tflite_files))

classes_path = dataset_root / 'classes.json'

files.download(str(tflite_files[-1]))
files.download(str(classes_path))
files.download(str(best_pt))